# 03 — Backtest & Regression: Forward Returns by Quartile

**Concentric Value Model · Empirical Validation · DBA Research**

**Research question Q-A:** *Do CVM top-quartile firms deliver higher risk-adjusted forward returns than bottom-quartile firms over 2016–2025?*

**Specification:** Forward 1-year total return regressed on CVM composite (or quartile dummies), with **all five controls**:
- Sector fixed effects
- Country fixed effects
- Log market cap (size factor)
- Market beta (CAPM-style)
- Regime dummies: COVID-2020 and 2022-2023 inflation/hiking

**Standard errors:** clustered by `as_of` quarter (accounts for cross-sectional return correlation within quarters).

**Outputs to Google Drive:**
- `quartile_portfolio_returns.csv` — Q1/Q2/Q3/Q4 mean returns per quarter
- `quartile_spread_series.csv` — Q1 minus Q4 time series
- `regression_continuous.csv` — full continuous-composite regression
- `regression_quartile_dummies.csv` — quartile-dummy regression (headline)
- `framework_alpha_summary.csv` — single-row result for thesis abstract
- `framework_alpha_by_period.csv` — sub-period robustness
- 2 figures (quartile growth paths, sub-period alpha)

## Step 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/cvm-research'
# Resolve the study window from the metadata nb01 wrote — never hardcode the
# filename, so changing START_YEAR/END_YEAR in nb01 can't silently desync.
import json as _json
with open(f'{WORK_DIR}/output/panel_metadata.json') as _f:
    _meta = _json.load(_f)
_window = _meta['window']  # e.g. '2016-2025'
_ys, _ye = _window.split('-')
PANEL_FILE = f"{WORK_DIR}/output/panel_pit_{_ys}_{_ye}.parquet"
SCORED_FILE = f"{WORK_DIR}/output/scored_panel_{_ys}_{_ye}.parquet"
print(f'Study window from metadata: {_window}')
scored_path = SCORED_FILE
assert os.path.exists(scored_path), 'Run 02_pca_analysis.ipynb first (it saves the scored panel).'

REPO_PATH = 'FelixDLanger/cvm-research'
import sys
!pip install statsmodels --quiet
!git clone https://github.com/{REPO_PATH}.git /content/cvm-research 2>/dev/null || (cd /content/cvm-research && git pull)
sys.path.insert(0, '/content/cvm-research')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from cvm_research.scoring import DIMENSIONS, LAYERS
from cvm_research.stats import (
    build_quartile_portfolios, quartile_spread_series, add_regime_dummies,
    run_panel_regression, run_quartile_dummy_regression,
    annualized_sharpe, regression_table_summary, sig_stars,
)
import cvm_research
print(f'cvm_research v{cvm_research.__version__}')

## Step 2 — Load scored panel and prepare for regression

In [ ]:
scored = pd.read_parquet(scored_path)
print(f'Loaded {len(scored):,} firm-quarter observations.')

# Add regime dummies
scored = add_regime_dummies(scored, date_col='as_of')
print()
print('Regime composition:')
print(f'  COVID regime (2020):       {scored["covid_regime"].sum():,} obs ({100*scored["covid_regime"].mean():.1f}%)')
print(f'  Inflation regime (2022-23): {scored["inflation_regime"].sum():,} obs ({100*scored["inflation_regime"].mean():.1f}%)')
print()
print('Coverage of regression inputs:')
for col in ['composite', 'forward_return_pct', 'market_cap', 'beta', 'sector', 'country']:
    n_present = scored[col].notna().sum() if col in scored.columns else 0
    print(f'  {col:22s}: {n_present:5d} / {len(scored):5d} ({100*n_present/len(scored):4.1f}%)')

## Step 3 — Quartile portfolio descriptives

In [ ]:
qport = build_quartile_portfolios(scored, return_col='forward_return_pct')
qport_pivot = qport.pivot(index='as_of', columns='quartile', values='mean_return')

# Full-sample summary
print('Mean forward 1-yr return by quartile (full sample):')
overall = qport.groupby('quartile')['mean_return'].agg(['mean','std','count']).round(2)
overall['sharpe_annualised'] = overall.apply(
    lambda r: round(r['mean']/r['std']*np.sqrt(4), 3) if r['std'] > 0 else np.nan, axis=1)
print(overall)
print()

spread = quartile_spread_series(qport)
ann_mean = spread['Q1_minus_Q4'].mean() * 4
ann_vol = spread['Q1_minus_Q4'].std() * np.sqrt(4)
print(f'Annualised Q1-minus-Q4 spread:')
print(f'  Mean:       {ann_mean:+.2f}%')
print(f'  Volatility: {ann_vol:.2f}%')
if ann_vol > 0:
    print(f'  Sharpe:     {ann_mean/ann_vol:+.3f}')

## Step 4 — Visualise quartile portfolio growth paths

In [ ]:
# Cumulative growth: convert quarterly % returns to growth factors, then cumprod
quartile_paths = qport_pivot.copy().sort_index()
for q in ['Q1','Q2','Q3','Q4']:
    if q in quartile_paths.columns:
        quartile_paths[f'{q}_growth'] = (1 + quartile_paths[q]/100).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
colors = {'Q1':'#4a7c3a', 'Q2':'#a8814a', 'Q3':'#888777', 'Q4':'#a14b3a'}
for q in ['Q1','Q2','Q3','Q4']:
    col = f'{q}_growth'
    if col in quartile_paths.columns:
        ax.plot(quartile_paths.index, quartile_paths[col], label=q,
                color=colors[q], linewidth=2)
ax.axhline(1.0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Growth (1.0 = starting value)')
ax.set_title('Quartile Portfolio Growth Paths\n1-Year Forward Total Return, Compounded Quarterly')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/output/quartile_growth_paths.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Headline regression (continuous composite + all 5 controls)

In [ ]:
# Specification: forward_return ~ composite + C(sector) + C(country)
#                                  + log_mcap + beta + covid_regime + inflation_regime
# Cluster-robust SE by as_of quarter
result_continuous = run_panel_regression(
    scored,
    return_col='forward_return_pct',
    composite_col='composite',
    cluster_by='as_of',
    verbose=True,
)
print(result_continuous.summary())

**Interpretation:**

- The `composite` coefficient is the marginal effect of a 1-point increase in CVM composite on forward 1-yr return (%), holding sector, country, size, beta, and regime constant.
- E.g. coef = 0.15 means: a 10-point higher CVM composite → +1.5% higher forward return, controls held fixed.
- p-value < 0.05 → statistically distinguishable from zero.
- Standard errors clustered by `as_of` quarter, accounting for cross-sectional correlation in forward returns within each quarter.

## Step 6 — Quartile dummy regression (Q1 vs Q4 — headline result)

In [ ]:
result_quartile = run_quartile_dummy_regression(
    scored,
    return_col='forward_return_pct',
    reference_quartile='auto',
    cluster_by='as_of',
    verbose=True,
)
print(result_quartile.summary())

In [ ]:
# Extract & interpret quartile coefficients (reference-aware)
ref_q = getattr(result_quartile, '_cvm_reference_quartile', 'Q4')
summary_df = regression_table_summary(result_quartile)

# All quartile dummy rows
qrows = summary_df[summary_df['variable'].str.contains('quartile', na=False)].copy()

print('='*64)
print(f'QUARTILE COEFFICIENTS  (reference group = {ref_q})')
print('='*64)
print(f'Each coefficient = mean forward-return difference vs {ref_q}, controls held fixed.\n')

import re
for _, row in qrows.iterrows():
    m = re.search(r'T\.(Q\d)', row['variable'])
    label = m.group(1) if m else row['variable']
    print(f'  {label} vs {ref_q}:  {row["coefficient"]:+7.3f}%   '
          f'(SE {row["std_error"]:.2f}, p={row["p_value"]:.4f} {sig_stars(row["p_value"])}, '
          f'95% CI [{row["ci_lower"]:+.2f}, {row["ci_upper"]:+.2f}])')

print('='*64)
print('INTERPRETATION GUIDE')
print('='*64)
print(f"""
The framework's directional claim is: HIGHER quartile (Q1 = High Conviction)
should outperform LOWER quartile (Q4 = Structural Concerns).

  • If Q1's coefficient is the LARGEST positive vs {ref_q}: framework ranks correctly.
  • If a LOWER quartile (Q2/Q3) beats Q1: the ranking is INVERTED in this sample
    — investigate before interpreting. Common causes: thin/regime-restricted
    sample, baseline-dominated scoring (check nb02 coverage), or genuine
    contrarian effect (rare; needs robustness checks across sub-periods).

Quote in the thesis with full uncertainty:
  "Relative to {ref_q} firms, Q1 firms showed X.XX% (95% CI [a,b], p=...)
   higher 1-year forward returns, controlling for sector, country, log market
   cap, beta, and regime."
""")


**Quoting this in your thesis abstract:**

> *"After controlling for sector, country, log market capitalisation, market beta, and regime dummies for COVID-19 (2020) and the 2022–2023 inflation/hiking cycle, top-quartile firms identified by the Concentric Value Model delivered X.XX% (95% CI: [a, b], p<X.XX) higher 1-year forward returns than bottom-quartile firms across the 2016–2025 sample period (N=X firm-quarter observations, standard errors clustered by quarter-end date)."*

Always quote the **point estimate with its 95% CI and p-value**, not the point estimate alone. This is the rigor a DBA examiner expects.

## Step 7 — Sub-period robustness

In [ ]:
periods = {
    'Pre-COVID (2016-2019)':     scored['as_of'].between('2016-01-01','2019-12-31'),
    'COVID era (2020-2021)':     scored['as_of'].between('2020-01-01','2021-12-31'),
    'Inflation+ (2022-2025)':    scored['as_of'].between('2022-01-01','2025-12-31'),
}

period_results = []
for name, mask in periods.items():
    sub = scored[mask].copy()
    n_obs = sub['forward_return_pct'].notna().sum()
    if n_obs < 50:
        print(f'  Skipping {name}: only {n_obs} observations')
        continue
    try:
        r = run_quartile_dummy_regression(sub, cluster_by='as_of', verbose=False)
        sd = regression_table_summary(r)
        q1 = sd[sd['variable'].str.contains('Q1', na=False) &
                sd['variable'].str.contains('quartile', na=False)]
        if len(q1) > 0:
            row = q1.iloc[0]
            period_results.append({
                'period': name,
                'n_obs': n_obs,
                'q1_alpha': round(row['coefficient'], 3),
                'std_err':  round(row['std_error'], 3),
                'p_value':  round(row['p_value'], 4),
                'sig':      sig_stars(row['p_value']),
                'ci_lower': round(row['ci_lower'], 3),
                'ci_upper': round(row['ci_upper'], 3),
            })
    except Exception as e:
        print(f'  {name} failed: {e}')

period_df = pd.DataFrame(period_results)
print()
print('Framework alpha across sub-periods:')
print(period_df.to_string(index=False))

In [ ]:
# Visualise sub-period alphas with error bars
if len(period_results) > 0:
    fig, ax = plt.subplots(figsize=(9, 5))
    x = range(len(period_df))
    ax.bar(x, period_df['q1_alpha'], color='#a8814a', alpha=0.85)
    ax.errorbar(x, period_df['q1_alpha'],
                yerr=[period_df['q1_alpha'] - period_df['ci_lower'],
                      period_df['ci_upper'] - period_df['q1_alpha']],
                fmt='none', color='black', capsize=5)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_xticks(x)
    ax.set_xticklabels(period_df['period'], rotation=0, fontsize=9)
    ax.set_ylabel('Framework Alpha (% per 1-yr forward)')
    ax.set_title('Q1-minus-Q4 Framework Alpha by Sub-Period\n(95% CI shown; controls: sector, country, log mcap, beta, regime)')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/output/framework_alpha_by_period.png', dpi=150, bbox_inches='tight')
    plt.show()

## Step 8 — Save all outputs for thesis appendix

In [ ]:
# Quartile portfolio data
qport.to_csv(f'{WORK_DIR}/output/quartile_portfolio_returns.csv', index=False)
spread.to_csv(f'{WORK_DIR}/output/quartile_spread_series.csv', index=False)

# Regression tables
regression_table_summary(result_continuous).to_csv(
    f'{WORK_DIR}/output/regression_continuous.csv', index=False)
regression_table_summary(result_quartile).to_csv(
    f'{WORK_DIR}/output/regression_quartile_dummies.csv', index=False)

# Single-row framework alpha summary
if len(q1_rows) > 0:
    row = q1_rows.iloc[0]
    pd.DataFrame({
        'metric': ['Framework Alpha (Q1 vs Q4)'],
        'coefficient_pct': [row['coefficient']],
        'std_error': [row['std_error']],
        't_stat': [row['t_stat']],
        'p_value': [row['p_value']],
        'significance': [sig_stars(row['p_value'])],
        'ci_lower': [row['ci_lower']],
        'ci_upper': [row['ci_upper']],
        'n_obs': [int(result_quartile.nobs)],
        'sample_period': ['2016-Q1 to 2025-Q4'],
        'standard_errors': ['cluster-robust by as_of quarter'],
        'controls': ['sector FE, country FE, log market_cap, beta, covid_regime, inflation_regime'],
    }).to_csv(f'{WORK_DIR}/output/framework_alpha_summary.csv', index=False)

# Sub-period table
if len(period_results) > 0:
    period_df.to_csv(f'{WORK_DIR}/output/framework_alpha_by_period.csv', index=False)

print('All regression outputs saved to:', f'{WORK_DIR}/output/')

## Summary — Q-A Answer

You now have the empirical answer to the headline DBA research question. The framework alpha is in `framework_alpha_summary.csv`.

**Three possible outcomes and how to frame each in the thesis:**

1. **Significant positive alpha (p<0.05):** The framework identifies firms that systematically outperform after rich controls. The thesis claims empirical validity for the methodology.

2. **Significant negative alpha:** The framework's top-quartile firms underperform. Itself an important finding — the framework as published would be a contrarian indicator.

3. **Not significant (p>0.10):** Most likely outcome at this sample size and rigor. The thesis argues the framework's contribution is as a **thinking tool** that organizes and structures the analysis of company quality across dimensions, independent of return-prediction claims. The PCA validation (notebook 02) speaks to the structural validity of the framework's dimensions; the regression speaks to its return-prediction power, and these can be reported separately.

**For viva defense**, the sub-period table shows robustness — examiners will ask "does this hold in normal markets, not just COVID?" The breakdown answers that directly.

**Combined with Q-B from notebook 02**, you have a complete empirical chapter: PCA confirms (or refines) the layer weights, AND quartile membership correlates (or doesn't) with forward returns after rigorous controls. That's a defensible chapter.